# FlightInsight — Great Expectations Suite

Ta zvezek ustvari `flights_suite` z relaxed range checks za naš dataset.

**Zaženi to celico samo ENKRAT** — suite se nato uporablja v `run_checkpoint.py`.

In [1]:
import great_expectations as gx
from great_expectations.core.expectation_configuration import ExpectationConfiguration

# Pomembno: poženemo iz mape gx/
context = gx.get_context()
print("GX context:", context)

GX context: {
  "anonymous_usage_statistics": {
    "explicit_id": true,
    "enabled": true,
    "usage_statistics_url": "https://stats.greatexpectations.io/great_expectations/v1/usage_statistics",
    "explicit_url": false,
    "data_context_id": "d88091c0-0ae4-482d-9041-4fb6e761f8d1"
  },
  "checkpoint_store_name": "checkpoint_store",
  "config_variables_file_path": "uncommitted/config_variables.yml",
  "config_version": 3.0,
  "data_docs_sites": {
    "local_site": {
      "class_name": "SiteBuilder",
      "show_how_to_buttons": true,
      "store_backend": {
        "class_name": "TupleFilesystemStoreBackend",
        "base_directory": "uncommitted/data_docs/local_site/"
      },
      "site_index_builder": {
        "class_name": "DefaultSiteIndexBuilder"
      }
    }
  },
  "datasources": {},
  "evaluation_parameter_store_name": "evaluation_parameter_store",
  "expectations_store_name": "expectations_store",
  "fluent_datasources": {},
  "include_rendered_content": {
    "expe

In [2]:
# 1. Datasource — naša data/ mapa
datasource = context.sources.add_or_update_pandas_filesystem(
    name="flights_data",
    base_directory="../data"
)

# 2. Data Asset — flights.csv v preprocessed/
data_asset = datasource.add_csv_asset(
    name="flights_csv",
    batching_regex=r"preprocessed/flights\.csv"
)

print("Datasource & Asset created")

Datasource & Asset created


In [3]:
# 3. Expectation Suite
suite_name = "flights_suite"
suite = context.add_or_update_expectation_suite(suite_name)

# Validator za podatke
batch_request = data_asset.build_batch_request()
validator = context.get_validator(
    batch_request=batch_request,
    expectation_suite_name=suite_name
)

# Pokaži statistiko
df = validator.active_batch.data.dataframe
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")
df.head()

Rows: 558,715
Columns: 26


,Year,Month,DayofMonth,DayOfWeek,FlightDate,Marketing_Airline_Network,Origin,OriginCityName,Dest,DestCityName,...,NASDelay,SecurityDelay,LateAircraftDelay,DepDelayMinutes,dep_hour,time_of_day,is_weekend,season,route,distance_group
0,2024,1,14,7,2024-01-14,UA,MHT,"Manchester, NH",EWR,"Newark, NJ",...,0.0,0.0,45.0,71.0,17,evening,1,winter,MHT-EWR,short
1,2024,1,14,7,2024-01-14,UA,IAD,"Washington, DC",EWR,"Newark, NJ",...,26.0,0.0,0.0,0.0,8,morning,1,winter,IAD-EWR,short
2,2024,1,14,7,2024-01-14,UA,EWR,"Newark, NJ",MHT,"Manchester, NH",...,12.0,0.0,0.0,74.0,15,afternoon,1,winter,EWR-MHT,short
3,2024,1,14,7,2024-01-14,UA,STL,"St. Louis, MO",ORD,"Chicago, IL",...,25.0,0.0,0.0,0.0,6,morning,1,winter,STL-ORD,short
4,2024,1,14,7,2024-01-14,UA,STL,"St. Louis, MO",IAD,"Washington, DC",...,0.0,0.0,0.0,33.0,13,afternoon,1,winter,STL-IAD,medium


## Pričakovanja (expectations)

Uporabljamo **relaxed range checks** — testiramo da so vrednosti znotraj smiselnih intervalov, ne točnih vrednosti. To preprečuje failed validation ob normalnih variacijah v podatkih.

In [4]:
# Tabela: vsaj 100k vrstic (varno, dataset ima 558k)
validator.expect_table_row_count_to_be_between(min_value=100_000, max_value=10_000_000)

# Stolpci: vsi pričakovani stolpci morajo biti prisotni
expected_columns = [
    "Year", "Month", "DayofMonth", "DayOfWeek", "FlightDate",
    "Marketing_Airline_Network",
    "Origin", "OriginCityName", "Dest", "DestCityName",
    "CRSDepTime", "CRSArrTime",
    "Distance", "CRSElapsedTime",
    "CarrierDelay", "WeatherDelay", "NASDelay",
    "SecurityDelay", "LateAircraftDelay",
    "DepDelayMinutes",
    "dep_hour", "time_of_day", "is_weekend", "season",
    "route", "distance_group"
]
validator.expect_table_columns_to_match_set(column_set=expected_columns)

Calculating Metrics:   0%|          | 0/1 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/2 [00:00<?, ?it/s]

{
  "success": true,
  "result": {
    "observed_value": [
      "Year",
      "Month",
      "DayofMonth",
      "DayOfWeek",
      "FlightDate",
      "Marketing_Airline_Network",
      "Origin",
      "OriginCityName",
      "Dest",
      "DestCityName",
      "CRSDepTime",
      "CRSArrTime",
      "Distance",
      "CRSElapsedTime",
      "CarrierDelay",
      "WeatherDelay",
      "NASDelay",
      "SecurityDelay",
      "LateAircraftDelay",
      "DepDelayMinutes",
      "dep_hour",
      "time_of_day",
      "is_weekend",
      "season",
      "route",
      "distance_group"
    ]
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

In [5]:
# === KLJUČNI STOLPCI: NE NULL ===
validator.expect_column_values_to_not_be_null("DepDelayMinutes")
validator.expect_column_values_to_not_be_null("Origin")
validator.expect_column_values_to_not_be_null("Dest")
validator.expect_column_values_to_not_be_null("Marketing_Airline_Network")
validator.expect_column_values_to_not_be_null("FlightDate")

Calculating Metrics:   0%|          | 0/6 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/6 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/6 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/6 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/6 [00:00<?, ?it/s]

{
  "success": true,
  "result": {
    "element_count": 558715,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": []
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

In [6]:
# === RANGE CHECKS — relaxed ===

# Target: DepDelayMinutes — 0 do 1440 (24h, brutalno dolge zamude)
validator.expect_column_values_to_be_between(
    "DepDelayMinutes", min_value=0, max_value=3500
)

# Razdalja: realne vrednosti za ZDA notranji promet
validator.expect_column_values_to_be_between(
    "Distance", min_value=10, max_value=6000
)

# Ura odhoda: 0-23
validator.expect_column_values_to_be_between(
    "dep_hour", min_value=0, max_value=23
)

# DayOfWeek: 1-7
validator.expect_column_values_to_be_between(
    "DayOfWeek", min_value=1, max_value=7
)

# Month: 1-12
validator.expect_column_values_to_be_between(
    "Month", min_value=1, max_value=12
)

# Year: 2018-2030 (širok razpon za prihodnost)
validator.expect_column_values_to_be_between(
    "Year", min_value=2018, max_value=2030
)

# is_weekend: samo 0 ali 1
validator.expect_column_values_to_be_in_set(
    "is_weekend", value_set=[0, 1]
)

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

{
  "success": true,
  "result": {
    "element_count": 558715,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": [],
    "missing_count": 0,
    "missing_percent": 0.0,
    "unexpected_percent_total": 0.0,
    "unexpected_percent_nonmissing": 0.0
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

In [7]:
# === KATEGORIČNE VREDNOSTI ===
validator.expect_column_values_to_be_in_set(
    "time_of_day", value_set=["morning", "afternoon", "evening", "night"]
)

validator.expect_column_values_to_be_in_set(
    "season", value_set=["winter", "spring", "summer", "fall"]
)

validator.expect_column_values_to_be_in_set(
    "distance_group", value_set=["short", "medium", "long", "very_long"]
)

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

{
  "success": true,
  "result": {
    "element_count": 558715,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": [],
    "missing_count": 0,
    "missing_percent": 0.0,
    "unexpected_percent_total": 0.0,
    "unexpected_percent_nonmissing": 0.0
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

In [8]:
# === FORMATI ===

# Origin/Dest morajo biti 3-črkovne IATA kode
validator.expect_column_value_lengths_to_equal("Origin", value=3)
validator.expect_column_value_lengths_to_equal("Dest", value=3)

# Marketing_Airline_Network: 2-3 znakov (AA, DL, WN, UA...)
validator.expect_column_value_lengths_to_be_between(
    "Marketing_Airline_Network", min_value=2, max_value=3
)

Calculating Metrics:   0%|          | 0/9 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/9 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/9 [00:00<?, ?it/s]

{
  "success": true,
  "result": {
    "element_count": 558715,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": [],
    "missing_count": 0,
    "missing_percent": 0.0,
    "unexpected_percent_total": 0.0,
    "unexpected_percent_nonmissing": 0.0
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

In [9]:
# === SHRANI SUITE ===
validator.save_expectation_suite(discard_failed_expectations=False)

# Pokaži povzetek
saved_suite = context.get_expectation_suite(suite_name)
print(f"Suite '{suite_name}' shranjen z {len(saved_suite.expectations)} pričakovanji.")

for exp in saved_suite.expectations:
    print(f"  - {exp.expectation_type} on {exp.kwargs.get('column', 'TABLE')}")

Suite 'flights_suite' shranjen z 20 pričakovanji.
  - expect_table_row_count_to_be_between on TABLE
  - expect_table_columns_to_match_set on TABLE
  - expect_column_values_to_not_be_null on DepDelayMinutes
  - expect_column_values_to_not_be_null on Origin
  - expect_column_values_to_not_be_null on Dest
  - expect_column_values_to_not_be_null on Marketing_Airline_Network
  - expect_column_values_to_not_be_null on FlightDate
  - expect_column_values_to_be_between on DepDelayMinutes
  - expect_column_values_to_be_between on Distance
  - expect_column_values_to_be_between on dep_hour
  - expect_column_values_to_be_between on DayOfWeek
  - expect_column_values_to_be_between on Month
  - expect_column_values_to_be_between on Year
  - expect_column_values_to_be_in_set on is_weekend
  - expect_column_values_to_be_in_set on time_of_day
  - expect_column_values_to_be_in_set on season
  - expect_column_values_to_be_in_set on distance_group
  - expect_column_value_lengths_to_equal on Origin
  - ex

In [10]:
# === USTVARI CHECKPOINT ===
checkpoint = context.add_or_update_checkpoint(
    name="flights_checkpoint",
    validations=[
        {
            "batch_request": batch_request,
            "expectation_suite_name": suite_name
        }
    ]
)
print("Checkpoint 'flights_checkpoint' ustvarjen.")

Checkpoint 'flights_checkpoint' ustvarjen.


In [11]:
# === TESTNO POŽENI CHECKPOINT ===
result = checkpoint.run()
context.build_data_docs()

if result['success']:
    print('✅ Vsa pričakovanja izpolnjena!')
else:
    print('❌ Nekatera pričakovanja niso izpolnjena.')
    
print(f"\nDocs ustvarjeni v: gx/uncommitted/data_docs/local_site/index.html")

Calculating Metrics:   0%|          | 0/114 [00:00<?, ?it/s]

✅ Vsa pričakovanja izpolnjena!

Docs ustvarjeni v: gx/uncommitted/data_docs/local_site/index.html
